In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from itertools import product
from sklearn.datasets import make_classification
import random
import plotly.io as pio
from sklearn.preprocessing import LabelEncoder
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OrdinalEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
import pandas as pd




# Dataset

In [2]:
def create_sample_dataset(n_samples, random_seed=42):
    random.seed(random_seed)

    n_small_categorical_features = 2
    n_large_categorical_features = 3
    n_numerical_features = 3

    n_categorical_features = n_small_categorical_features + n_large_categorical_features
    n_features = n_numerical_features + n_categorical_features
    columns = (
        [f'num_feature_{i}' for i in range(n_numerical_features)]
        + [f'cat_small_feature_{i}' for i in range(n_small_categorical_features)]
        + [f'cat_large_feature_{i}' for i in range(n_large_categorical_features)]
        + ['target']
    )

    # --- Dataset Creation ---
    X, y = make_classification(n_samples=n_samples, n_features=n_features, random_state=random_seed)
    y = y.reshape(-1, 1)
    df = pd.DataFrame(np.concatenate([X, y], axis=1), columns=columns)

    # --- Numerical Features ---

    # --- Small Categorical Features ---
    for i in range(n_small_categorical_features):
        col = f"cat_small_feature_{i}"
        n_bins = random.randint(2, 5)
        df[col] = pd.cut(df[col], bins=n_bins, labels=[f'Small_{j}' for j in range(1, n_bins + 1)])

    # --- Large Categorical Features ---
    for i in range(n_large_categorical_features):
        col = f"cat_large_feature_{i}"
        n_bins = random.randint(20, 50)
        df[col] = pd.cut(df[col], bins=n_bins, labels=[f'Large_{j}' for j in range(1, n_bins + 1)])

    return df


# Modeling without categorical features

In [4]:

# Step 1: Load dataset
df = create_sample_dataset(1000)

# Step 2: Define feature groups
target_col = 'target'
numerical_cols = [col for col in df.columns if col.startswith('num_')]

X = df[numerical_cols]
y = df[target_col]

# Step 3: Train/test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Step 4: Full pipeline with classifier
pipeline = Pipeline(steps=[
    ('classifier', RandomForestClassifier(random_state=42))
])

# Step 5: Fit and evaluate
pipeline.fit(X_train, y_train)
accuracy = pipeline.score(X_test, y_test)

print(f"Model accuracy: {accuracy:.3f}")

Model accuracy: 0.640


# Modeling with categorical features (A)

In [5]:

# ---------------------
# Create dataset
# ---------------------
df = create_sample_dataset(1000)
target_col = 'target'

categorical_cols = [col for col in df.columns if col.startswith('cat_')]
numerical_cols = [col for col in df.columns if col.startswith('num_')]
X = df[categorical_cols + numerical_cols]
y = df[target_col]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# ---------------------
# Build pipeline
# ---------------------
cat_encoder = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', cat_encoder)
])

numerical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='mean')),
    ('scaler', StandardScaler())
])

preprocessor = ColumnTransformer(transformers=[
    # ('num', numerical_transformer, numerical_cols),
    ('cat', categorical_transformer, categorical_cols)
])

pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(random_state=42))
])

# ---------------------
# Fit pipeline
# ---------------------
pipeline.fit(X_train, y_train)
display(pipeline)
accuracy = pipeline.score(X_test, y_test)
print(f"Model accuracy: {accuracy:.3f}")

# ---------------------
# Transform test data
# ---------------------

# Extract the fitted encoder from the fitted pipeline
# Important: access from .named_transformers_ after fitting
fitted_cat_pipeline = pipeline.named_steps['preprocessor'].named_transformers_['cat']
fitted_encoder = fitted_cat_pipeline.named_steps['encoder']

# Transform X_test via preprocessor
test_sample = X_test.head(1)
test_sample_transformed = pipeline.named_steps['preprocessor'].transform(test_sample)

# Extract categorical-encoded part of the transformed data
cat_encoded = test_sample_transformed
cat_encoded_df = pd.DataFrame(cat_encoded, columns=categorical_cols)

# Now safely inverse transform using the *fitted* encoder
cat_decoded = fitted_encoder.inverse_transform(cat_encoded)

# Convert back to DataFrame
decoded_df = pd.DataFrame(cat_decoded, columns=categorical_cols)

# ---------------------
# Print comparison
# ---------------------
print("Original categorical values:")
display(test_sample[categorical_cols].head())

print("\nEncoded values:")
display(cat_encoded_df[categorical_cols].head())

print("\nDecoded values after inverse_transform:")
display(decoded_df.head())


Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('cat',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('encoder',
                                                                   OrdinalEncoder(handle_unknown='use_encoded_value',
                                                                                  unknown_value=-1))]),
                                                  ['cat_small_feature_0',
                                                   'cat_small_feature_1',
                                                   'cat_large_feature_0',
                                                   'cat_large_feature_1',
                                                   'cat_large_feature_2'])])),
                ('classifier', RandomForestClassifier(random_state=42))])

Model accuracy: 0.800
Original categorical values:


,cat_small_feature_0,cat_small_feature_1,cat_large_feature_0,cat_large_feature_1,cat_large_feature_2
521,Small_2,Small_2,Large_24,Large_13,Large_14



Encoded values:


,cat_small_feature_0,cat_small_feature_1,cat_large_feature_0,cat_large_feature_1,cat_large_feature_2
0,1.0,1.0,16.0,4.0,5.0



Decoded values after inverse_transform:


,cat_small_feature_0,cat_small_feature_1,cat_large_feature_0,cat_large_feature_1,cat_large_feature_2
0,Small_2,Small_2,Large_24,Large_13,Large_14


# Modeling with categorical features (B)

In [ ]:

# ---------------------
# Create dataset
# ---------------------
df = create_sample_dataset(1000)
target_col = 'target'

categorical_cols = [col for col in df.columns if col.startswith('cat_')]
numerical_cols = [col for col in df.columns if col.startswith('num_')]
X = df[categorical_cols + numerical_cols]
y = df[target_col]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# ---------------------
# Build pipeline
# ---------------------
cat_encoder = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', cat_encoder)
])

numerical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='mean')),
    ('scaler', StandardScaler())
])

preprocessor = ColumnTransformer(transformers=[
    # ('num', numerical_transformer, numerical_cols),
    ('cat', categorical_transformer, categorical_cols)
])

pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(random_state=42))
])

# ---------------------
# Fit pipeline
# ---------------------
pipeline.fit(X_train, y_train)
display(pipeline)
accuracy = pipeline.score(X_test, y_test)
print(f"Model accuracy: {accuracy:.3f}")

# ---------------------
# Transform test data
# ---------------------

# Extract the fitted encoder from the fitted pipeline
# Important: access from .named_transformers_ after fitting
fitted_cat_pipeline = pipeline.named_steps['preprocessor'].named_transformers_['cat']
fitted_encoder = fitted_cat_pipeline.named_steps['encoder']

# Transform X_test via preprocessor
test_sample = X_test.head(1)
test_sample_transformed = pipeline.named_steps['preprocessor'].transform(test_sample)

# Extract categorical-encoded part of the transformed data
cat_encoded = test_sample_transformed
cat_encoded


Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('cat',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('encoder',
                                                                   OrdinalEncoder(handle_unknown='use_encoded_value',
                                                                                  unknown_value=-1))]),
                                                  ['cat_small_feature_0',
                                                   'cat_small_feature_1',
                                                   'cat_large_feature_0',
                                                   'cat_large_feature_1',
                                                   'cat_large_feature_2'])])),
                ('classifier', RandomForestClassifier(random_state=42))])

Model accuracy: 0.800


array([[ 1.,  1., 16.,  4.,  5.]])